# E2 + E2.5 — decodability and spectral share, n=14 and n=16

Clean run of the experiments that survived
`AAAI_n14_E2_ridge_probe_exploration.ipynb`, executed identically on both
datasets. That notebook is the record of the two probe failures and their
diagnoses (RidgeCV LOO-GCV underflow -> cv=5; power-sum rank collapse ->
lstsq); the fixes are baked in here from the top. No protocol changes.

Per-dataset differences are correctness only: cospectral mate pairs are
discovered by spectrum hashing (not hardcoded from n=14), expected
enumeration counts asserted per n, cubic identities parameterized
(tr(A^2) = 3n), C6 computed in-consumer where the producer didn't store
it, paths and vector widths per n.

Runtime, stated up front: the n=16 E2 table is the long pole —
RidgeCV(cv=5) is 425 fits per target on 4060 x 65536; expect hours.
Everything else is minutes. Config: canonical angles, r=1 (locked).

In [1]:
import pickle

import numpy as np
import networkx as nx

from collections import defaultdict
from itertools import combinations

from numpy.linalg import eigvalsh, svd
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr, pearsonr
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline

from tqdm.notebook import tqdm

In [2]:
SEED = 0
ALPHAS = np.logspace(-14, 2, 17)   # wide floor: exploration nb, LOO-GCV record
N_SPLITS = 5
NS = (14, 16)

BASES = {
    14: "/kaggle/input/notebooks/lukemiller1987/n14-aaai-dataset/",
    16: "/kaggle/input/notebooks/lukemiller1987/n16-aaai-dataset/",
}
EXPECTED_CUBIC_COUNTS = {10: 19, 12: 85, 14: 509, 16: 4060}

## Load both datasets, build target arrays

Defensive target loading as in the exploration notebook: read producer
fields if present, compute from graph6/adj_mat otherwise. C6 was never a
producer field at n=14 and may not be at n=16 — computed in-consumer via
`nx.simple_cycles(length_bound=6)`, same validated counter as the
producer's 3..5 counts.

In [3]:
def load_dataset(n):
    with open(BASES[n] + f"n{n}_data_dict.pkl", 'rb') as f:
        data_dict = pickle.load(f)
    keys = sorted(data_dict)
    assert len(keys) == EXPECTED_CUBIC_COUNTS[n], (
        f'expected {EXPECTED_CUBIC_COUNTS[n]} graphs at n={n}, got {len(keys)}')

    graphs = {k: nx.from_graph6_bytes(data_dict[k]['graph6'].encode())
              for k in keys}

    def get_or_compute(field, fn, progress=False):
        if field in data_dict[keys[0]]:
            return np.array([data_dict[k][field] for k in keys], dtype=float)
        it = tqdm(keys, desc=f'n={n} {field}') if progress else keys
        return np.array([fn(k) for k in it], dtype=float)

    def c6_count(k):
        return sum(1 for c in nx.simple_cycles(graphs[k], length_bound=6)
                   if len(c) == 6)

    D = {
        'keys': keys,
        'vectors': np.vstack([data_dict[k]['exact_vector'] for k in keys]),
        'num_triangles': np.array([data_dict[k]['num_triangles'] for k in keys]),
        'num_4_cycles':  np.array([data_dict[k]['num_4_cycles']  for k in keys]),
        'num_5_cycles':  np.array([data_dict[k]['num_5_cycles']  for k in keys]),
        'num_6_cycles':  get_or_compute('num_6_cycles', c6_count,
                                        progress=True).astype(int),
        'girth':    get_or_compute('girth',    lambda k: nx.girth(graphs[k])),
        'diameter': get_or_compute('diameter', lambda k: nx.diameter(graphs[k])),
    }
    if 'spectrum' in data_dict[keys[0]]:
        D['spectra'] = np.stack([data_dict[k]['spectrum'] for k in keys])
    else:
        D['spectra'] = np.stack([np.sort(eigvalsh(
            data_dict[k]['adj_mat'])) for k in keys])
    D['spectral_gap'] = D['spectra'][:, -1] - D['spectra'][:, -2]

    D['targets'] = {
        'triangles':    D['num_triangles'].astype(float),
        '4-cycles':     D['num_4_cycles'].astype(float),
        '5-cycles':     D['num_5_cycles'].astype(float),
        '6-cycles':     D['num_6_cycles'].astype(float),
        'girth':        D['girth'],
        'diameter':     D['diameter'],
        'spectral_gap': D['spectral_gap'],
    }
    return D

DATA = {}
for n in NS:
    DATA[n] = load_dataset(n)
    print(f"n={n}: {len(DATA[n]['keys'])} graphs, "
          f"vectors {DATA[n]['vectors'].shape}, "
          f"C6 range {DATA[n]['num_6_cycles'].min()}.."
          f"{DATA[n]['num_6_cycles'].max()}")

n=14 num_6_cycles:   0%|          | 0/509 [00:00<?, ?it/s]

n=14: 509 graphs, vectors (509, 16384), C6 range 0..28


n=16 num_6_cycles:   0%|          | 0/4060 [00:00<?, ?it/s]

n=16: 4060 graphs, vectors (4060, 65536), C6 range 0..24


## Probe machinery — exploration protocol, verbatim

`ridge_probe`: RidgeCV with explicit nested CV (`cv=5`; the closed-form
LOO selector underflows in the near-interpolating regime — see
exploration notebook). Full vectors go in raw. `lstsq_probe`: SVD
min-norm least squares for the collinear power-sum family. Splits are
frozen per n and saved with the E2 results — E3 must reuse them
verbatim.

In [4]:
def ridge_probe(X, y, splits, alphas=ALPHAS, standardize=False):
    """Per-fold: RidgeCV fit on train (alpha via nested cv=5), R2 on test."""
    scores, chosen = [], []
    for tr, te in splits:
        if standardize:
            model = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas, cv=5))
        else:
            model = RidgeCV(alphas=alphas, cv=5)
        model.fit(X[tr], y[tr])
        scores.append(r2_score(y[te], model.predict(X[te])))
        a = model[-1].alpha_ if standardize else model.alpha_
        chosen.append(a)
    return np.array(scores), chosen

def lstsq_probe(X, y, splits):
    """CV R2 via min-norm least squares — stable under collinear features."""
    scores = []
    for tr, te in splits:
        mu, sd = X[tr].mean(0), X[tr].std(0)
        Xtr = np.column_stack([np.ones(len(tr)), (X[tr] - mu) / sd])
        Xte = np.column_stack([np.ones(len(te)), (X[te] - mu) / sd])
        w, *_ = np.linalg.lstsq(Xtr, y[tr], rcond=None)
        scores.append(r2_score(y[te], Xte @ w))
    return np.array(scores)

SPLITS = {}
for n in NS:
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    SPLITS[n] = [(tr, te) for tr, te in kf.split(DATA[n]['vectors'])]

## E2 — headline decodability table, both n

Seven targets (C6 promoted into the table; its n=14 row was 0.485 +-
0.326 in exploration — layer-four fraying — and n=16 is where the plan
says layer four gets adjudicated). Expect chosen alphas at/near the grid
floor: exact arithmetic, noiseless targets. The n=16 pass is the
multi-hour cell.

In [5]:
E2_RESULTS = {}
for n in NS:
    vectors, targets = DATA[n]['vectors'], DATA[n]['targets']
    E2_RESULTS[n] = {}
    print(f'\n=== n={n} ===')
    print(f'{"target":>13}   R2 mean +- sd    per-fold R2'
          f'                          alphas chosen')
    for name, y in targets.items():
        sc, al = ridge_probe(vectors, y, SPLITS[n])
        E2_RESULTS[n][name] = {'r2_folds': sc, 'r2_mean': sc.mean(),
                               'r2_sd': sc.std(), 'alphas_chosen': al}
        folds = ' '.join(f'{s:+.3f}' for s in sc)
        print(f'{name:>13}:  {sc.mean():+.3f} +- {sc.std():.3f}   '
              f'[{folds}]   {sorted(set(al))}')


=== n=14 ===
       target   R2 mean +- sd    per-fold R2                          alphas chosen
    triangles:  +1.000 +- 0.000   [+1.000 +1.000 +1.000 +1.000 +1.000]   [np.float64(1e-12)]
     4-cycles:  +0.998 +- 0.002   [+0.997 +0.999 +0.996 +0.999 +0.999]   [np.float64(1e-13)]
     5-cycles:  +0.928 +- 0.092   [+0.968 +0.948 +0.746 +0.987 +0.988]   [np.float64(1e-13), np.float64(1e-12)]
     6-cycles:  +0.485 +- 0.326   [+0.515 +0.537 -0.132 +0.712 +0.794]   [np.float64(1e-14), np.float64(1e-13), np.float64(1e-12)]
        girth:  +0.993 +- 0.014   [+0.965 +1.000 +1.000 +1.000 +1.000]   [np.float64(1e-14)]
     diameter:  +0.548 +- 0.058   [+0.606 +0.501 +0.476 +0.626 +0.532]   [np.float64(1e-12), np.float64(1e-10), np.float64(1e-09)]
 spectral_gap:  +0.944 +- 0.010   [+0.937 +0.937 +0.935 +0.958 +0.955]   [np.float64(1e-13), np.float64(1e-12)]

=== n=16 ===
       target   R2 mean +- sd    per-fold R2                          alphas chosen
    triangles:  +1.000 +- 0.000   [+1.0

In [6]:
for n in NS:
    out = {
        'splits': [(tr.tolist(), te.tolist()) for tr, te in SPLITS[n]],
        'seed': SEED, 'n_splits': N_SPLITS, 'alphas_grid': ALPHAS,
        'results': E2_RESULTS[n],
        'note': 'E3 must reuse these exact splits for the matched-CV comparison.',
    }
    with open(f'/kaggle/working/n{n}_e2_ridge_probe.pkl', 'wb') as f:
        pickle.dump(out, f)
    print(f'saved n{n}_e2_ridge_probe.pkl')

saved n14_e2_ridge_probe.pkl
saved n16_e2_ridge_probe.pkl


## Power-sum check and moment ladder, both n

Three-scalar check first (the exploration headline), then the nested
ladder k = 2..8 under `lstsq_probe`. n=14 findings this replicates or
extends: tri = purity alone (0.999); C4 saturates at exactly k=5 at its
full-vector ceiling; C5 absent from all eight (<= 0.11). The open n=16
questions: does the k=5 edge for C4 hold, and does C6 enter the ladder
anywhere.

In [7]:
LADDER = {}
K = 8
for n in NS:
    vectors, targets = DATA[n]['vectors'], DATA[n]['targets']
    P3 = np.stack([(vectors ** 2).sum(axis=1),
                   (vectors ** 3).sum(axis=1),
                   (vectors ** 4).sum(axis=1)], axis=1)
    print(f'\n=== n={n} ===')
    print(f'{"target":>13}   full-vector R2     power-sum R2 (3 features)')
    for name in ['triangles', '4-cycles', '5-cycles']:
        sc, _ = ridge_probe(P3, targets[name], SPLITS[n], standardize=True)
        full = E2_RESULTS[n][name]
        print(f'{name:>13}:  {full["r2_mean"]:+.3f} +- {full["r2_sd"]:.3f}     '
              f'{sc.mean():+.3f} +- {sc.std():.3f}')

    PS = np.stack([(vectors ** k).sum(axis=1) for k in range(2, K + 1)], axis=1)
    DATA[n]['PS'] = PS
    LADDER[n] = {}
    print(f'\n{"k up to":>8}   tri      C4       C5       C6')
    for j in range(1, K):
        row = {t: lstsq_probe(PS[:, :j], targets[t], SPLITS[n]).mean()
               for t in ['triangles', '4-cycles', '5-cycles', '6-cycles']}
        LADDER[n][j + 1] = row
        print(f'{j+1:>8}   ' + '  '.join(
            f'{row[t]:+.3f}' for t in row))

    with open(f'/kaggle/working/n{n}_moment_ladder.pkl', 'wb') as f:
        pickle.dump({'ladder_r2': LADDER[n],
                     'probe': 'lstsq_probe (SVD, rcond=None)',
                     'features': 'standardized power sums, k=2..8, nested',
                     'seed': SEED,
                     'splits_from': f'n{n}_e2_ridge_probe.pkl'}, f)
    print(f'saved n{n}_moment_ladder.pkl')


=== n=14 ===
       target   full-vector R2     power-sum R2 (3 features)
    triangles:  +1.000 +- 0.000     +1.000 +- 0.000
     4-cycles:  +0.998 +- 0.002     +0.754 +- 0.043
     5-cycles:  +0.928 +- 0.092     +0.064 +- 0.066

 k up to   tri      C4       C5       C6
       2   +0.999  +0.001  +0.022  +0.144
       3   +1.000  +0.677  +0.066  +0.184
       4   +1.000  +0.755  +0.069  +0.182
       5   +1.000  +0.997  +0.045  +0.189
       6   +1.000  +0.997  +0.101  +0.183
       7   +1.000  +0.997  +0.107  +0.195
       8   +1.000  +0.997  +0.106  +0.198
saved n14_moment_ladder.pkl

=== n=16 ===
       target   full-vector R2     power-sum R2 (3 features)
    triangles:  +1.000 +- 0.000     +1.000 +- 0.000
     4-cycles:  +1.000 +- 0.000     +0.750 +- 0.022
     5-cycles:  +0.982 +- 0.015     +0.056 +- 0.008

 k up to   tri      C4       C5       C6
       2   +0.999  +0.003  +0.024  +0.100
       3   +1.000  +0.691  +0.056  +0.142
       4   +1.000  +0.752  +0.058  +0.143
      

In [8]:
# C5 interaction test, both n: degree-2 polynomial over Sum p^2..p^5.
# n=14 verdict was flat (0.14 +- 0.10): C5 is not quadratic in low moments;
# carrier is finer symmetric-function content. Replication check at n=16.
for n in NS:
    Q = PolynomialFeatures(degree=2, include_bias=False).fit_transform(
        DATA[n]['PS'][:, :4])
    sc = lstsq_probe(Q, DATA[n]['targets']['5-cycles'], SPLITS[n])
    print(f'n={n}:  C5 ~ deg-2 poly of 4 power sums ({Q.shape[1]} feats): '
          f'R2 = {sc.mean():+.3f} +- {sc.std():.3f}')

n=14:  C5 ~ deg-2 poly of 4 power sums (14 feats): R2 = +0.143 +- 0.103
n=16:  C5 ~ deg-2 poly of 4 power sums (14 feats): R2 = +0.132 +- 0.015


## E2.5 — spectral decomposition, both n

Identical to exploration: cubic identities asserted (tr(A^2) = 3n,
tr(A^3) = 6 tri — triangles route through the spectrum by identity, so
the emb~tri|spectrum partial is subsumed); per-PC decomposition against
cycles / six spectral moments / full spectrum; C6 suppression probe on
PC2. n=14 findings on record: spectral moments ~57-59% of variance, PC2
(15.3%) non-spectral with C6's non-spectral residue at ~0.21 of it.

In [9]:
E25 = {}
for n in NS:
    D = DATA[n]
    vectors, spectra, targets = D['vectors'], D['spectra'], D['targets']

    assert np.allclose((spectra ** 2).sum(axis=1), 3 * n)
    assert np.allclose((spectra ** 3).sum(axis=1), 6 * D['num_triangles'])

    SPEC_KS = range(3, 9)
    M = np.stack([(spectra ** k).mean(axis=1) for k in SPEC_KS], axis=1)
    SPEC_FULL = spectra[:, :-1]          # drop constant Perron eigenvalue

    Xc = vectors - vectors.mean(axis=0)
    U, S, Vt = svd(Xc, full_matrices=False)
    var_ratio = S**2 / (S**2).sum()
    pc_scores = Xc @ Vt[:6].T
    D['pc_scores'], D['var_ratio'] = pc_scores, var_ratio
    CYC = np.stack([D['num_triangles'], D['num_4_cycles'],
                    D['num_5_cycles']], axis=1).astype(float)

    spec_decomp = {}
    print(f'\n=== n={n} ===')
    print('PC    var%    R2 cycles   R2 moments   R2 full-spec   R2 mom+cyc')
    for pc in range(6):
        y = pc_scores[:, pc]
        row = {'var_ratio': var_ratio[pc],
               'r2_cycles':   lstsq_probe(CYC, y, SPLITS[n]).mean(),
               'r2_moments':  lstsq_probe(M, y, SPLITS[n]).mean(),
               'r2_spectrum': lstsq_probe(SPEC_FULL, y, SPLITS[n]).mean(),
               'r2_both':     lstsq_probe(np.hstack([M, CYC]), y,
                                          SPLITS[n]).mean()}
        spec_decomp[pc + 1] = row
        print(f'PC{pc+1}   {row["var_ratio"]*100:5.1f}   '
              f'{row["r2_cycles"]:+.3f}      {row["r2_moments"]:+.3f}       '
              f'{row["r2_spectrum"]:+.3f}         {row["r2_both"]:+.3f}')

    # C6 suppression probe on PC2 (n=14: alone -0.03, moments+C6 0.21)
    y2 = pc_scores[:, 1]
    c6 = D['num_6_cycles'].astype(float)
    r_c6  = lstsq_probe(c6.reshape(-1, 1), y2, SPLITS[n]).mean()
    r_mc6 = lstsq_probe(np.column_stack([M, c6]), y2, SPLITS[n]).mean()
    r_m   = spec_decomp[2]['r2_moments']
    print(f'PC2 ~ C6 alone: {r_c6:+.3f}   PC2 ~ moments+C6: {r_mc6:+.3f}   '
          f'(moments alone: {r_m:+.3f})')

    E25[n] = {'pc_decomposition': spec_decomp, 'spec_ks': list(SPEC_KS),
              'pc2_c6': {'alone': r_c6, 'with_moments': r_mc6}}


=== n=14 ===
PC    var%    R2 cycles   R2 moments   R2 full-spec   R2 mom+cyc
PC1    51.2   +0.962      +0.962       +0.954         +0.962
PC2    15.3   -0.002      +0.060       +0.195         +0.060
PC3     7.8   +0.128      +0.119       +0.097         +0.119
PC4     7.7   +0.802      +0.811       +0.809         +0.811
PC5     5.3   -0.014      -0.017       +0.010         -0.017
PC6     2.6   -0.033      -0.040       -0.065         -0.040
PC2 ~ C6 alone: -0.028   PC2 ~ moments+C6: +0.210   (moments alone: +0.060)

=== n=16 ===
PC    var%    R2 cycles   R2 moments   R2 full-spec   R2 mom+cyc
PC1    50.7   +0.957      +0.958       +0.957         +0.958
PC2    15.1   +0.021      +0.052       +0.275         +0.052
PC3     8.4   +0.013      +0.013       +0.016         +0.013
PC4     7.6   +0.923      +0.933       +0.927         +0.933
PC5     5.5   +0.003      +0.010       +0.076         +0.010
PC6     2.4   -0.004      -0.000       +0.016         -0.000
PC2 ~ C6 alone: -0.003   PC2 ~ mom

## Cospectral mates — discovery, separation, tracer, reference class

Mates discovered per n by spectrum hashing (n=14 must recover
(79,458), (201,203), (234,239) — acts as the regression check on this
notebook). Then the three yardsticks from exploration, in order: raw
separations with global percentile (moment-axis yardstick, kept for
continuity), the truncation tracer at full precision (head-weighting),
and the count-identical reference class (fine-structure yardstick — the
one that matters). Distances between sorted vectors are W1 between
probability multisets; print full precision — one-decimal formatting
hid the tracer effect once already.

In [10]:
COSPEC = {}
for n in NS:
    D = DATA[n]
    keys, vectors, spectra = D['keys'], D['vectors'], D['spectra']
    tri, c4, c5 = D['num_triangles'], D['num_4_cycles'], D['num_5_cycles']

    spec_hash = [tuple(np.round(s, 8)) for s in spectra]
    mates = defaultdict(list)
    for k, h in zip(keys, spec_hash):
        mates[h].append(k)
    groups = [v for v in mates.values() if len(v) > 1]
    pairs = [p for g in groups for p in combinations(g, 2)]
    print(f'\n=== n={n}: {len(groups)} cospectral group(s), '
          f'{len(pairs)} pair(s) among {len(keys)} graphs ===')

    all_d = pdist(vectors, metric='cityblock')
    records = []
    for i, j in pairs:
        ci = (int(tri[i]), int(c4[i]), int(c5[i]))
        cj = (int(tri[j]), int(c4[j]), int(c5[j]))
        assert ci == cj, 'cospectral cubic mates must share counts'
        d = np.abs(vectors[i] - vectors[j]).sum()
        pct = (all_d < d).mean() * 100
        records.append({'pair': (i, j), 'L1': d,
                        'global_percentile': pct, 'counts': ci})
        print(f'  pair ({i},{j})  counts {ci}  L1 = {d:.4e}  '
              f'({pct:.5f} global pctile)')
    COSPEC[n] = {'pairs': pairs, 'records': records}
    del all_d


=== n=14: 3 cospectral group(s), 3 pair(s) among 509 graphs ===
  pair (79,458)  counts (2, 1, 2)  L1 = 3.9081e-07  (0.06110 global pctile)
  pair (201,203)  counts (1, 3, 3)  L1 = 1.6919e-05  (5.81734 global pctile)
  pair (234,239)  counts (1, 2, 4)  L1 = 5.8681e-08  (0.00077 global pctile)

=== n=16: 41 cospectral group(s), 43 pair(s) among 4060 graphs ===
  pair (1,5)  counts (0, 6, 0)  L1 = 1.6797e-05  (6.32421 global pctile)
  pair (24,35)  counts (0, 5, 0)  L1 = 6.4840e-08  (0.00235 global pctile)
  pair (82,885)  counts (0, 1, 4)  L1 = 2.6601e-07  (0.02853 global pctile)
  pair (83,884)  counts (0, 1, 3)  L1 = 2.6598e-07  (0.02852 global pctile)
  pair (106,632)  counts (2, 5, 0)  L1 = 1.6767e-05  (6.23243 global pctile)
  pair (259,268)  counts (1, 4, 1)  L1 = 1.0483e-07  (0.00631 global pctile)
  pair (282,1494)  counts (0, 2, 2)  L1 = 3.4032e-08  (0.00019 global pctile)
  pair (320,3059)  counts (1, 1, 3)  L1 = 2.7850e-07  (0.03391 global pctile)
  pair (457,516)  counts (1

In [11]:
# Truncation tracer, full precision. n=14 record: two of three pairs gain
# 1-2 orders of magnitude in relative standing as k shrinks (growth below
# k ~ 200) — truncation preferentially retains the head-weighted residue.
for n in NS:
    D = DATA[n]
    vectors, pairs = D['vectors'], COSPEC[n]['pairs']
    if not pairs:
        print(f'n={n}: no cospectral pairs — tracer skipped')
        continue
    full = vectors.shape[1]
    trunc_ks = [k for k in (25, 50, 100, 200, 400, 1000, 4000) if k < full]
    trunc_ks.append(full)
    tracer = {}
    print(f'\n=== n={n} ===')
    print(f'{"k":>6}   ' + '   '.join(f'({i},{j})' for i, j in pairs))
    for k in trunc_ks:
        Vk = vectors[:, :k]
        all_d = pdist(Vk, metric='cityblock')
        pcts = [(all_d < np.abs(Vk[i] - Vk[j]).sum()).mean() * 100
                for i, j in pairs]
        tracer[k] = pcts
        print(f'{k:>6}   ' + '   '.join(f'{p:.5f}' for p in pcts))
    COSPEC[n]['tracer'] = {'ks': trunc_ks, 'percentiles': tracer}
    del all_d


=== n=14 ===
     k   (79,458)   (201,203)   (234,239)
    25   0.73094   5.71369   0.19646
    50   0.76265   18.29510   0.10519
   100   0.68762   8.65523   0.04254
   200   0.08508   7.97147   0.03017
   400   0.08740   6.94661   0.00077
  1000   0.06265   5.95579   0.00155
  4000   0.06110   5.81734   0.00000
 16384   0.06110   5.81734   0.00077

=== n=16 ===
     k   (1,5)   (24,35)   (82,885)   (83,884)   (106,632)   (259,268)   (282,1494)   (320,3059)   (457,516)   (476,3741)   (479,967)   (494,568)   (582,3061)   (582,3800)   (3061,3800)   (583,3067)   (586,1502)   (594,601)   (710,711)   (817,1130)   (936,940)   (1034,1116)   (1035,1123)   (1075,1086)   (1181,3517)   (1371,1792)   (1496,1570)   (1505,1506)   (1710,1734)   (1898,3951)   (2121,3077)   (2209,2226)   (2276,3578)   (2428,2431)   (2693,3900)   (3047,3520)   (3058,3799)   (3090,3525)   (3162,3248)   (3514,3515)   (3602,4005)   (3672,3689)   (3693,3902)
    25   1.36453   0.11216   0.03916   0.01289   1.39079   0.181

In [12]:
# Count-identical reference class: pairs matched on (tri, C4, C5), whose
# separations are entirely non-moment fine structure. n=14 record: mates
# rank 14th-93rd pctile here vs 0.0-5.8 globally — the global ranking was
# the moment axis; this is the yardstick that means something.
for n in NS:
    D = DATA[n]
    vectors, pairs = D['vectors'], COSPEC[n]['pairs']
    if not pairs:
        print(f'n={n}: no cospectral pairs — reference class skipped')
        continue
    tri, c4, c5 = D['num_triangles'], D['num_4_cycles'], D['num_5_cycles']

    strata3 = defaultdict(list)
    for k in D['keys']:
        strata3[(tri[k], c4[k], c5[k])].append(k)
    ci_pairs = [p for m in strata3.values() if len(m) > 1
                for p in combinations(m, 2)]
    n_all = len(D['keys']) * (len(D['keys']) - 1) // 2
    print(f'\n=== n={n}: {len(ci_pairs)} count-identical pairs '
          f'(of {n_all} total) ===')
    for i, j in pairs:
        c = (tri[i], c4[i], c5[i])
        m = len(strata3[c])
        print(f'  mate ({i},{j}): stratum {tuple(int(x) for x in c)}, '
              f'{m} members, {m * (m - 1) // 2} own-stratum pairs')

    full = vectors.shape[1]
    ci_result = {}
    print(f'{"k":>6}   pair          pctile among count-identical pairs')
    for k in [25, 100, full]:
        Vk = vectors[:, :k]
        d = np.array([np.abs(Vk[i] - Vk[j]).sum() for i, j in ci_pairs])
        row = {}
        for i, j in pairs:
            dij = np.abs(Vk[i] - Vk[j]).sum()
            row[(i, j)] = (d < dij).mean() * 100
            print(f'{k:>6}   ({i:4d},{j:4d})   {row[(i, j)]:9.3f}')
        ci_result[k] = row
    COSPEC[n]['count_identical'] = {
        'n_ci_pairs': len(ci_pairs), 'mate_percentiles': ci_result,
        'strata_sizes': {tuple(int(x) for x in k): len(v)
                         for k, v in strata3.items()}}

    with open(f'/kaggle/working/n{n}_cospectral_analysis.pkl', 'wb') as f:
        pickle.dump({k: v for k, v in COSPEC[n].items()}, f)
    print(f'saved n{n}_cospectral_analysis.pkl')


=== n=14: 715 count-identical pairs (of 129286 total) ===
  mate (79,458): stratum (2, 1, 2), 5 members, 10 own-stratum pairs
  mate (201,203): stratum (1, 3, 3), 9 members, 36 own-stratum pairs
  mate (234,239): stratum (1, 2, 4), 7 members, 21 own-stratum pairs
     k   pair          pctile among count-identical pairs
    25   (  79, 458)      22.937
    25   ( 201, 203)      78.741
    25   ( 234, 239)      13.986
   100   (  79, 458)      24.476
   100   ( 201, 203)      91.888
   100   ( 234, 239)       7.692
 16384   (  79, 458)      10.909
 16384   ( 201, 203)      92.727
 16384   ( 234, 239)       0.000
saved n14_cospectral_analysis.pkl

=== n=16: 52181 count-identical pairs (of 8239770 total) ===
  mate (1,5): stratum (0, 6, 0), 9 members, 36 own-stratum pairs
  mate (24,35): stratum (0, 5, 0), 11 members, 55 own-stratum pairs
  mate (82,885): stratum (0, 1, 4), 26 members, 325 own-stratum pairs
  mate (83,884): stratum (0, 1, 3), 9 members, 36 own-stratum pairs
  mate (106,6

In [13]:
# E2.5 persistence, both n
for n in NS:
    with open(f'/kaggle/working/n{n}_e25_spectral.pkl', 'wb') as f:
        pickle.dump(E25[n], f)
    print(f'saved n{n}_e25_spectral.pkl')

saved n14_e25_spectral.pkl
saved n16_e25_spectral.pkl


## Results

(Written after the run: the E4 scale comparison — every n=14 number
against its n=16 replication — plus the three open n=16 questions: does
C4's k=5 edge hold, does C6 enter the table/ladder as layer four with
(tri,C4,C5) joint strata now populated, and where do the n=16 cospectral
mates sit in their count-identical class.)